# 12B — Point-in-Time S&P 500 SEC Corpus

This notebook resolves SEC identifiers, inventories point-in-time 10-K/10-Q filings, reuses all existing caches, and constructs the retrieval queue. It makes **no OpenAI calls** and never opens return outcomes. Document downloading is separately gated.

## 1. Environment and verified upstream handoff

In [ ]:
from pathlib import Path
import concurrent.futures as cf, gzip, hashlib, importlib, json, os, shutil, sys, threading, time
import numpy as np
import pandas as pd
from IPython.display import display
HERE=Path.cwd().resolve()
if HERE.name=='sp500_critical_minerals': EXT=HERE; GEO=HERE.parent; ROOT=HERE.parents[1]
elif HERE.name=='geopolitical_exposure': GEO=HERE; ROOT=HERE.parent; EXT=GEO/'sp500_critical_minerals'
elif HERE.name=='Asset_Embeddings': ROOT=HERE; GEO=ROOT/'geopolitical_exposure'; EXT=GEO/'sp500_critical_minerals'
else: ROOT=(HERE/'Projects'/'Asset_Embeddings').resolve(); GEO=ROOT/'geopolitical_exposure'; EXT=GEO/'sp500_critical_minerals'
CONFIG=EXT/'config'; RAW=EXT/'data'/'raw'; PROCESSED=EXT/'data'/'processed'; RESULTS=EXT/'results'; MANIFESTS=EXT/'manifests'
for p in [RAW,PROCESSED,RESULTS,MANIFESTS]: p.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(GEO/'src'))
import document_contracts,edgar_pipeline,quarterly_sec_extraction
importlib.reload(document_contracts);importlib.reload(edgar_pipeline);importlib.reload(quarterly_sec_extraction)
from document_contracts import normalize_document,validate_documents
from edgar_pipeline import download_company_tickers,filing_metadata_for_cik,get_text,html_to_text
def sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
design_path=CONFIG/'sp500_experiment_design_v1.json';design=json.loads(design_path.read_text());design_hash=sha256(design_path)
upstream=json.loads((MANIFESTS/'universe_manifest.json').read_text())
assert upstream['all_gates_passed'] and not upstream['outcomes_accessed'] and not upstream['api_calls_made']
bridge_path=PROCESSED/'sp500_point_in_time_identifier_bridge.parquet';bridge=pd.read_parquet(bridge_path)
vintages=pd.DataFrame(design['vintages']);vintages['cutoff']=pd.to_datetime(vintages.cutoff)
print('Verified universe manifest:',upstream['sha256']);print('Return outcomes opened:',False);print('OpenAI calls made:',False)

## 2. Resolve the remaining CIKs using the official SEC ticker file

CCM remains primary. Exact normalized ticker matching is used only for firms missing a CCM-derived CIK, and every fallback is preserved in the audit.

In [ ]:
SEC_USER_AGENT=os.getenv('SEC_USER_AGENT','').strip()
if '@' not in SEC_USER_AGENT: SEC_USER_AGENT=input('SEC user agent (research name and university email): ').strip()
if '@' not in SEC_USER_AGENT: raise ValueError('SEC_USER_AGENT must identify the researcher and contain an email address.')
sec_reference=GEO/'documents'/'raw'/'sec_reference'/'company_tickers_exchange.parquet'
sec=download_company_tickers(sec_reference,SEC_USER_AGENT).copy()
def clean_ticker(x): return str(x).upper().strip().replace('.','-').replace('/','-') if pd.notna(x) else pd.NA
sec['ticker_clean']=sec.ticker.map(clean_ticker);sec_counts=sec.groupby('ticker_clean').cik.nunique()
firm_bridge=(bridge.sort_values(['vintage','permco','cik'],na_position='last').drop_duplicates(['vintage','permco']).copy())
firm_bridge['ticker_clean']=firm_bridge.ticker.map(clean_ticker);firm_bridge['cik_source']=np.where(firm_bridge.cik.notna(),'ccm_compustat','unresolved')
fallback=firm_bridge.cik.isna() & firm_bridge.ticker_clean.map(sec_counts).eq(1).fillna(False)
sec_unique=sec[sec.ticker_clean.map(sec_counts).eq(1)].drop_duplicates('ticker_clean').set_index('ticker_clean').cik
firm_bridge.loc[fallback,'cik']=firm_bridge.loc[fallback,'ticker_clean'].map(sec_unique).astype('Int64')
firm_bridge.loc[fallback,'cik_source']='sec_unique_exact_ticker'
firm_bridge['cik']=pd.to_numeric(firm_bridge.cik,errors='coerce').astype('Int64')
firm_bridge.to_parquet(PROCESSED/'sp500_firm_quarter_sec_bridge.parquet',index=False)
resolution=(firm_bridge.groupby('vintage',as_index=False).agg(firms=('permco','nunique'),resolved_ciks=('cik','count'),unique_ciks=('cik','nunique'),fallback_links=('cik_source',lambda x:int((x=='sec_unique_exact_ticker').sum()))));resolution['coverage']=resolution.resolved_ciks/resolution.firms
display(resolution);display(firm_bridge[firm_bridge.cik_source!='ccm_compustat'][['vintage','permco','ticker','comnam','cik','cik_source']])

## 3. Configure resumable SEC acquisition

Metadata acquisition is enabled because it is free and mostly cached. Full document downloading remains disabled until the exact plan in Section 6 is reviewed.

In [ ]:
DOWNLOAD_SEC_METADATA=True
DOWNLOAD_SEC_DOCUMENTS=False
MAX_DOCUMENT_WORKERS=4
MIN_SECONDS_BETWEEN_SEC_REQUESTS=0.13
print('Metadata enabled:',DOWNLOAD_SEC_METADATA);print('Document download enabled:',DOWNLOAD_SEC_DOCUMENTS);print('Maximum aggregate SEC request rate:',round(1/MIN_SECONDS_BETWEEN_SEC_REQUESTS,2),'requests/second')

## 4. Acquire and checkpoint filing metadata

Each CIK submission response is stored independently. Interrupted execution resumes from the cache. Metadata are requested once per unique CIK, not once per quarter.

In [ ]:
metadata_path=PROCESSED/'sp500_sec_filing_metadata.parquet';error_path=RESULTS/'sec_metadata_errors.parquet';submission_cache=GEO/'documents'/'raw'/'sec_submissions'
cik_firms=(firm_bridge.dropna(subset=['cik']).sort_values(['cik','permco']).drop_duplicates('cik')[['cik','permco']])
if metadata_path.exists(): filing_metadata=pd.read_parquet(metadata_path);print('Loaded checkpointed filing metadata.')
elif not DOWNLOAD_SEC_METADATA: raise RuntimeError('Enable DOWNLOAD_SEC_METADATA and rerun Sections 3–4.')
else:
 frames=[];errors=[]
 for position,row in enumerate(cik_firms.itertuples(index=False),1):
  try:
   f=filing_metadata_for_cik(int(row.cik),SEC_USER_AGENT,design['filing_forms'],design['filing_start'],str(vintages.cutoff.max().date()),submission_cache)
   if len(f): f['reference_permco']=int(row.permco);frames.append(f)
  except Exception as exc: errors.append({'cik':int(row.cik),'permco':int(row.permco),'error_type':type(exc).__name__,'error':str(exc)})
  if position%25==0 or position==len(cik_firms): print(f'Metadata {position}/{len(cik_firms)}; errors={len(errors)}')
 filing_metadata=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame();pd.DataFrame(errors).to_parquet(error_path,index=False)
 if errors: raise RuntimeError(f'{len(errors)} metadata failures remain; rerun safely after inspection.')
 filing_metadata.to_parquet(metadata_path,index=False)
filing_metadata['filingDate']=pd.to_datetime(filing_metadata.filingDate);filing_metadata['reportDate']=pd.to_datetime(filing_metadata.reportDate)
display(pd.DataFrame([{'unique_ciks':cik_firms.cik.nunique(),'metadata_filings':len(filing_metadata),'firms_with_metadata':filing_metadata.cik.nunique(),'ten_k':int(filing_metadata.form.eq('10-K').sum()),'ten_q':int(filing_metadata.form.eq('10-Q').sum()),'latest_filing':filing_metadata.filingDate.max()}]))

## 5. Construct the four point-in-time filing selections

Each firm-quarter uses the latest available 10-K plus the latest 10-Q filed after that annual anchor. The same accession serving several vintages is processed and billed only once.

In [ ]:
selected=[]
for v in vintages.itertuples(index=False):
 qbridge=firm_bridge[firm_bridge.vintage==v.label].dropna(subset=['cik']).copy()
 for r in qbridge.itertuples(index=False):
  available=filing_metadata[(filing_metadata.cik.astype('Int64')==int(r.cik)) & filing_metadata.filingDate.le(v.cutoff)]
  annual=available[available.form.eq('10-K')].sort_values(['filingDate','accessionNumber'])
  if annual.empty: continue
  a=annual.iloc[-1];selected.append({'vintage':v.label,'cutoff':v.cutoff,'permco':int(r.permco),'cik':int(r.cik),'document_role':'structural_anchor',**a.to_dict()})
  quarterly=available[available.form.eq('10-Q') & available.filingDate.gt(a.filingDate)].sort_values(['filingDate','accessionNumber'])
  if len(quarterly): q=quarterly.iloc[-1];selected.append({'vintage':v.label,'cutoff':v.cutoff,'permco':int(r.permco),'cik':int(r.cik),'document_role':'quarterly_update',**q.to_dict()})
vintage_documents=pd.DataFrame(selected).drop_duplicates(['vintage','permco','accessionNumber']);vintage_documents['document_id']=vintage_documents.accessionNumber.astype(str)
unique_documents=vintage_documents.sort_values(['filingDate','accessionNumber']).drop_duplicates('accessionNumber').copy()
vintage_documents.to_parquet(PROCESSED/'sp500_vintage_document_map.parquet',index=False);unique_documents.to_parquet(PROCESSED/'sp500_unique_selected_documents.parquet',index=False)
coverage=(vintage_documents.groupby('vintage',as_index=False).agg(firms=('permco','nunique'),documents=('document_id','nunique'),ten_k=('form',lambda x:int((x=='10-K').sum())),ten_q=('form',lambda x:int((x=='10-Q').sum())),latest_available_date=('filingDate','max')));coverage=resolution[['vintage','firms']].rename(columns={'firms':'universe_firms'}).merge(coverage,on='vintage',how='left');coverage['firm_coverage']=coverage.firms/coverage.universe_firms
coverage.to_parquet(RESULTS/'vintage_document_coverage.parquet',index=False);display(coverage)

## 6. Exact download and cache-reuse plan

Stop here on the first run. Review how many unique filings already exist locally and how many new SEC requests are required before enabling Section 7.

In [ ]:
new_html_dir=RAW/'sec_filings';new_html_dir.mkdir(parents=True,exist_ok=True)
cache_dirs=[new_html_dir,GEO/'documents'/'raw'/'sec_filings',GEO/'text_expansion_v2'/'raw'/'sec_filings',GEO/'quarterly_sec_pilot'/'data'/'raw'/'sec_filings',GEO/'quarterly_sec_pilot'/'sector_supplement'/'data'/'raw'/'sec_filings']
html_index={}
for directory in cache_dirs:
 if directory.exists():
  for pattern in ['*.html','*.htm','*.html.gz','*.htm.gz']:
   for p in directory.glob(pattern): html_index.setdefault(p.stem,p)
unique_documents['cached_path']=unique_documents.accessionNumber.astype(str).map(lambda x:str(html_index[x]) if x in html_index else pd.NA)
unique_documents['already_cached']=unique_documents.cached_path.notna()
download_plan=unique_documents[['document_id','cik','form','filingDate','source_uri','already_cached','cached_path']].copy();download_plan.to_parquet(PROCESSED/'sec_document_download_plan.parquet',index=False)
size_lookup=unique_documents.set_index('document_id')['size'].pipe(pd.to_numeric,errors='coerce')
uncached_bytes=size_lookup.reindex(download_plan.loc[~download_plan.already_cached,'document_id']).sum()
plan_summary=pd.DataFrame([{'unique_documents':len(download_plan),'already_cached':int(download_plan.already_cached.sum()),'new_sec_downloads':int((~download_plan.already_cached).sum()),'cache_reuse_rate':download_plan.already_cached.mean(),'estimated_uncached_uncompressed_gb':uncached_bytes/1e9,'raw_storage_format':'gzip_html','document_download_enabled':DOWNLOAD_SEC_DOCUMENTS,'api_calls':0}])
plan_summary.to_parquet(RESULTS/'document_download_plan_summary.parquet',index=False);display(plan_summary)
if not DOWNLOAD_SEC_DOCUMENTS: print('STOP POINT: inspect this table, then set DOWNLOAD_SEC_DOCUMENTS=True in Section 3 to authorize filing downloads.')

## 7. Concurrent, rate-limited document download

Four workers reduce latency, while a shared limiter keeps aggregate traffic below the declared SEC request rate. Every completed HTML file is checkpointed immediately.

In [11]:
if not DOWNLOAD_SEC_DOCUMENTS: raise RuntimeError('Document downloading is disabled. Review Section 6, then enable it in Section 3.')
import gzip, requests
rate_lock=threading.Lock();last_request=[0.0]
headers=edgar_pipeline.sec_headers(SEC_USER_AGENT)
def fetch_one(row):
 destination=new_html_dir/f'{row.document_id}.html.gz'
 if destination.exists(): return {'document_id':row.document_id,'status':'checkpoint','path':str(destination),'error':None}
 partial=destination.with_suffix(destination.suffix+'.part')
 for attempt in range(5):
  try:
   with rate_lock:
    wait=MIN_SECONDS_BETWEEN_SEC_REQUESTS-(time.monotonic()-last_request[0])
    if wait>0: time.sleep(wait)
    last_request[0]=time.monotonic()
   response=requests.get(row.source_uri,headers=headers,timeout=120,stream=True);response.raise_for_status()
   with gzip.open(partial,'wb',compresslevel=6) as handle:
    for chunk in response.iter_content(chunk_size=1024*1024):
     if chunk: handle.write(chunk)
   partial.replace(destination)
   return {'document_id':row.document_id,'status':'downloaded','path':str(destination),'error':None}
  except Exception as exc:
   if partial.exists(): partial.unlink()
   if attempt==4: return {'document_id':row.document_id,'status':'failed','path':pd.NA,'error':f'{type(exc).__name__}: {exc}'}
   time.sleep(2**attempt)
todo=download_plan[~download_plan.already_cached]
download_rows=[]
with cf.ThreadPoolExecutor(max_workers=MAX_DOCUMENT_WORKERS) as pool:
 futures={pool.submit(fetch_one,row):row.document_id for row in todo.itertuples(index=False)}
 for number,future in enumerate(cf.as_completed(futures),1):
  download_rows.append(future.result())
  if number%25==0 or number==len(futures): print(f'Downloaded/checkpointed {number}/{len(futures)} new files')
download_audit=pd.DataFrame(download_rows);download_audit.to_parquet(RESULTS/'document_download_audit.parquet',index=False)
failures=download_audit[download_audit.status.eq('failed')] if len(download_audit) else download_audit
display(download_audit.status.value_counts(dropna=False).rename_axis('status').reset_index(name='documents') if len(download_audit) else pd.DataFrame([{'status':'all_cached','documents':len(download_plan)}]))
if len(failures): raise RuntimeError(f'{len(failures)} downloads failed. Successful files are checkpointed; rerun safely.')

Downloaded/checkpointed 25/1880 new files
Downloaded/checkpointed 50/1880 new files
Downloaded/checkpointed 75/1880 new files
Downloaded/checkpointed 100/1880 new files
Downloaded/checkpointed 125/1880 new files
Downloaded/checkpointed 150/1880 new files
Downloaded/checkpointed 175/1880 new files
Downloaded/checkpointed 200/1880 new files
Downloaded/checkpointed 225/1880 new files
Downloaded/checkpointed 250/1880 new files
Downloaded/checkpointed 275/1880 new files
Downloaded/checkpointed 300/1880 new files
Downloaded/checkpointed 325/1880 new files
Downloaded/checkpointed 350/1880 new files
Downloaded/checkpointed 375/1880 new files
Downloaded/checkpointed 400/1880 new files
Downloaded/checkpointed 425/1880 new files
Downloaded/checkpointed 450/1880 new files
Downloaded/checkpointed 475/1880 new files
Downloaded/checkpointed 500/1880 new files
Downloaded/checkpointed 525/1880 new files
Downloaded/checkpointed 550/1880 new files
Downloaded/checkpointed 575/1880 new files
Downloaded/che

,status,documents
0,downloaded,1880


## 8. Normalize, validate, and hash the unique corpus

In [12]:
retrieval_path=PROCESSED/'sp500_retrieval_records.jsonl'
index_rows=[];queue_rows=[];normalization_errors=[]
retrieval_handle=retrieval_path.open('w',encoding='utf-8')
for number,row in enumerate(unique_documents.itertuples(index=False),1):
 try:
  accession=str(row.accessionNumber);cached=Path(row.cached_path) if pd.notna(row.cached_path) else new_html_dir/f'{accession}.html.gz'
  if cached.suffix=='.gz': html=gzip.open(cached,'rt',encoding='utf-8',errors='replace').read()
  else: html=cached.read_text(encoding='utf-8',errors='replace')
  text=html_to_text(html);record=normalize_document({'document_id':accession,'firm_id':str(int(row.reference_permco)),'document_type':str(row.form),'document_date':pd.Timestamp(row.reportDate).date().isoformat(),'available_date':pd.Timestamp(row.filingDate).date().isoformat(),'source':'SEC EDGAR','source_uri':row.source_uri,'text':text})
  passages=quarterly_sec_extraction.retrieve_passages(record['text'])
  retrieval={'document_id':record['document_id'],'firm_id':record['firm_id'],'document_type':record['document_type'],'available_date':record['available_date'],'content_sha256':record['content_sha256'],'passages':passages,'retrieved_characters':sum(len(p['text']) for p in passages)}
  retrieval_handle.write(json.dumps(retrieval,ensure_ascii=False)+'\n')
  index_rows.append({'document_id':record['document_id'],'firm_id':record['firm_id'],'document_type':record['document_type'],'document_date':record['document_date'],'available_date':record['available_date'],'source_uri':record['source_uri'],'content_sha256':record['content_sha256'],'characters':len(record['text']),'raw_path':str(cached)})
  queue_rows.append({'document_id':record['document_id'],'firm_id':record['firm_id'],'document_type':record['document_type'],'available_date':record['available_date'],'content_sha256':record['content_sha256'],'passage_count':len(passages),'retrieved_characters':retrieval['retrieved_characters'],'approx_input_tokens':int(np.ceil(retrieval['retrieved_characters']/4))})
 except Exception as exc: normalization_errors.append({'document_id':str(row.accessionNumber),'error_type':type(exc).__name__,'error':str(exc)})
 if number%100==0 or number==len(unique_documents): print(f'Normalized {number}/{len(unique_documents)}; errors={len(normalization_errors)}')
retrieval_handle.close()
pd.DataFrame(normalization_errors).to_parquet(RESULTS/'normalization_errors.parquet',index=False)
if normalization_errors: raise RuntimeError('Normalization failures remain; inspect the saved audit.')
document_index=pd.DataFrame(index_rows);document_index.to_parquet(PROCESSED/'sp500_document_index.parquet',index=False)
queue=pd.DataFrame(queue_rows);queue.to_parquet(PROCESSED/'sp500_retrieval_audit.parquet',index=False)
display(pd.DataFrame([{'documents':len(document_index),'firms':document_index.firm_id.nunique(),'median_characters':document_index.characters.median(),'stored_characters':document_index.characters.sum()}]))

Normalized 100/2080; errors=0
Normalized 200/2080; errors=0
Normalized 300/2080; errors=0
Normalized 400/2080; errors=0
Normalized 500/2080; errors=0
Normalized 600/2080; errors=0
Normalized 700/2080; errors=0
Normalized 800/2080; errors=0
Normalized 900/2080; errors=0
Normalized 1000/2080; errors=0
Normalized 1100/2080; errors=0
Normalized 1200/2080; errors=0
Normalized 1300/2080; errors=0
Normalized 1400/2080; errors=0
Normalized 1500/2080; errors=0
Normalized 1600/2080; errors=0
Normalized 1700/2080; errors=0
Normalized 1800/2080; errors=0
Normalized 1900/2080; errors=0
Normalized 2000/2080; errors=0
Normalized 2080/2080; errors=0


,documents,firms,median_characters,stored_characters
0,2080,506,228636.0,628479762


## 9. Frozen retrieval and exact API-volume forecast

Only retrieval-positive documents enter the API queue. Retrieval-negative documents are retained as `not identified in supplied text`, not asserted to have zero true exposure.

In [13]:
if 'queue' not in globals(): queue=pd.read_parquet(PROCESSED/'sp500_retrieval_audit.parquet')
api_queue=queue[queue.passage_count.gt(0)].copy();api_queue.to_parquet(PROCESSED/'sp500_api_queue.parquet',index=False);queue.to_parquet(PROCESSED/'sp500_retrieval_audit.parquet',index=False)
forecast=pd.DataFrame([{'documents':len(queue),'retrieval_positive_documents':len(api_queue),'retrieval_positive_rate':len(api_queue)/len(queue) if len(queue) else np.nan,'queued_firms':api_queue.firm_id.nunique(),'approx_input_tokens':api_queue.approx_input_tokens.sum(),'assumed_output_tokens_at_800_each':len(api_queue)*800,'guarded_first_batch':design['decision_gates']['api_pilot_documents'],'openai_calls_made':False}])
forecast.to_parquet(RESULTS/'exact_api_volume_forecast.parquet',index=False);display(forecast)
print('Cost formula: input_tokens / 1e6 × selected-model input price + output_tokens / 1e6 × selected-model output price.')

,documents,retrieval_positive_documents,retrieval_positive_rate,queued_firms,approx_input_tokens,assumed_output_tokens_at_800_each,guarded_first_batch,openai_calls_made
0,2080,923,0.44375,356,1124898,738400,25,False


Cost formula: input_tokens / 1e6 × selected-model input price + output_tokens / 1e6 × selected-model output price.


## 10. Corpus gates and immutable manifest

In [15]:
information=vintage_documents.merge(document_index[['document_id','content_sha256','characters']],on='document_id',how='left',validate='many_to_one');information['point_in_time_valid']=pd.to_datetime(information.filingDate).le(pd.to_datetime(information.cutoff));information.to_parquet(PROCESSED/'sp500_firm_quarter_information_panel.parquet',index=False)
panel=(information.groupby('vintage',as_index=False).agg(firms=('permco','nunique'),documents=('document_id','nunique'),missing_documents=('content_sha256',lambda x:int(x.isna().sum())),cutoff_violations=('point_in_time_valid',lambda x:int((~x).sum()))));panel=resolution[['vintage','firms']].rename(columns={'firms':'universe_firms'}).merge(panel,on='vintage');panel['coverage']=panel.firms/panel.universe_firms
g=design['decision_gates'];gates=pd.DataFrame([{'gate':'cik_coverage','value':resolution.coverage.min(),'threshold':g['minimum_cik_mapping_rate'],'passed':bool(resolution.coverage.min()>=g['minimum_cik_mapping_rate'])},{'gate':'filing_coverage','value':panel.coverage.min(),'threshold':g['minimum_filing_coverage_rate'],'passed':bool(panel.coverage.min()>=g['minimum_filing_coverage_rate'])},{'gate':'all_documents_present','value':panel.missing_documents.sum(),'threshold':0,'passed':bool(panel.missing_documents.sum()==0)},{'gate':'point_in_time_integrity','value':panel.cutoff_violations.sum(),'threshold':0,'passed':bool(panel.cutoff_violations.sum()==0)},{'gate':'outcomes_unopened','value':0,'threshold':0,'passed':True},{'gate':'openai_calls_not_made','value':0,'threshold':0,'passed':True}]);gates.to_parquet(MANIFESTS/'corpus_decision_gates.parquet',index=False);display(panel);display(gates)
manifest={'stage':'sp500_sec_corpus','design_sha256':design_hash,'upstream_manifest_sha256':upstream['sha256'],'bridge_sha256':sha256(PROCESSED/'sp500_firm_quarter_sec_bridge.parquet'),'document_map_sha256':sha256(PROCESSED/'sp500_vintage_document_map.parquet'),'document_index_sha256':sha256(PROCESSED/'sp500_document_index.parquet'),'retrieval_sha256':sha256(retrieval_path),'api_queue_sha256':sha256(PROCESSED/'sp500_api_queue.parquet'),'unique_documents':len(document_index),'api_queue_documents':len(api_queue),'all_gates_passed':bool(gates.passed.all()),'outcomes_accessed':False,'openai_calls_made':False}
manifest['sha256']=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest();(MANIFESTS/'corpus_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
print('All corpus gates passed:',manifest['all_gates_passed']);print('Manifest:',manifest['sha256'])

,vintage,universe_firms,firms,documents,missing_documents,cutoff_violations,coverage
0,2024q1,500,495,599,0,0,0.990000
1,2024q2,500,493,968,0,0,0.986000
2,2024q3,501,492,952,0,0,0.982036
3,2024q4,500,492,944,0,0,0.984000


,gate,value,threshold,passed
0,cik_coverage,0.996000,0.95,True
1,filing_coverage,0.982036,0.95,True
2,all_documents_present,0.000000,0.00,True
3,point_in_time_integrity,0.000000,0.00,True
4,outcomes_unopened,0.000000,0.00,True
5,openai_calls_not_made,0.000000,0.00,True


All corpus gates passed: True
Manifest: 6b76eb01daec4804251090ee6511517112c3f0750123dae5e244bc6cb182ef9c
